# 04 Dense Retrieval PoC (`GermanQuAD` + `intfloat/multilingual-e5-large`)

Dieses Notebook implementiert den ersten reproduzierbaren Dense-Retrieval-Pilot auf den **normalisierten** Eingaben unter `code/output/germanquad/`.

Fokus:
- Retrieval-only ohne Query-Rewriting und ohne Generierung
- genau ein erstes Embedding-Modell: `intfloat/multilingual-e5-large`
- identische Evaluationslogik und identische `k`-Werte wie im BM25-PoC
- Dokument-Ebene statt Chunk-Ebene, weil die aktuellen qrels auf `doc_id` zeigen
- `text` als Dokumentrepräsentation

## Methodik und Metriken

**Eingaben**
- `queries.normalized.jsonl`
- `docs.normalized.jsonl`
- `qrels.normalized.jsonl`

**Warum dieses Setup?**
- `GermanQuAD` ist für den ersten Dense-Pilot technisch kontrollierbarer als `GerLeRB`.
- Die qrels referenzieren Dokumente, nicht Chunks. Deshalb wird noch nicht gechunkt.
- Die Dense-Auswertung nutzt dieselben Cutoffs `k = [1, 5, 10]` wie der BM25-PoC, damit die Ergebnisse direkt vergleichbar bleiben.

**Modell und Similarity**
- Modell: `intfloat/multilingual-e5-large`
- Query-Format: `query: <text>`
- Dokument-Format: `passage: <text>`
- Similarity-Metrik: Cosine Similarity auf normalisierten Embeddings

**Metriken**
- `Recall@k`
- `Success@k`
- `MRR@k`
- optional `Precision@k`

Bei den aktuell lokal vorliegenden qrels gibt es pro Query genau ein bekanntes relevantes Dokument. Deshalb sind `Recall@k` und `Success@k` numerisch gleich, werden aber getrennt ausgewiesen.

In [1]:
from __future__ import annotations

import json
import platform
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib.metadata import version as package_version
from pathlib import Path
from typing import Any

import pandas as pd
import sys

MODEL_NAME = "intfloat/multilingual-e5-large"
DATASET_NAME = "germanquad"
K_VALUES = [1, 5, 10]
RELEVANCE_THRESHOLD = 1
BATCH_SIZE = 32
NORMALIZE_EMBEDDINGS = True
SAMPLE_QUERY_COUNT = 5
PYTHON_VERSION = platform.python_version()

ROOT = Path.cwd().resolve()
if not (ROOT / "code").exists() and ROOT.name == "code":
    ROOT = ROOT.parent

CODE_DIR = ROOT / "code"
OUTPUT_DIR = CODE_DIR / "output"
DENSE_OUTPUT_DIR = OUTPUT_DIR / "dense_e5"
PYLIB_DIR = CODE_DIR / "pylib"

DENSE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(PYLIB_DIR) not in sys.path:
    sys.path.insert(0, str(PYLIB_DIR))

DATASET_PATHS = {
    "queries": OUTPUT_DIR / DATASET_NAME / "queries.normalized.jsonl",
    "docs": OUTPUT_DIR / DATASET_NAME / "docs.normalized.jsonl",
    "qrels": OUTPUT_DIR / DATASET_NAME / "qrels.normalized.jsonl",
}

print(f"ROOT={ROOT}")
print(f"python={PYTHON_VERSION}")
print(f"dataset={DATASET_NAME}")
print(f"model={MODEL_NAME}")
print(f"output={DENSE_OUTPUT_DIR}")


ROOT=<PROJEKT>
python=3.9.6
dataset=germanquad
model=intfloat/multilingual-e5-large
output=<PROJEKT>/code/output/dense_e5


In [2]:
try:
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Paket 'sentence-transformers' ist nicht installiert. Bitte installiere es vor der Ausführung mit: pip install sentence-transformers"
    ) from exc

from local_dataset_io import load_jsonl

SENTENCE_TRANSFORMERS_VERSION = package_version("sentence-transformers")


<PROJEKT>/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
<PROJEKT>/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def compose_doc_text(doc: dict[str, Any]) -> str:
    return str(doc.get("text") or "").strip()


def format_query(text: str) -> str:
    return f"query: {text.strip()}"


def format_passage(text: str) -> str:
    return f"passage: {text.strip()}"


@dataclass
class DatasetBundle:
    name: str
    split_values: list[str]
    queries: list[dict[str, Any]]
    docs: list[dict[str, Any]]
    qrels: list[dict[str, Any]]


def load_dataset(name: str, paths: dict[str, Path]) -> DatasetBundle:
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Fehlende normalisierte Datei(en) für Datensatz '"
            f"{name}': {missing}. Erwartet werden Dateien unter code/output/<dataset>/."
        )

    queries = load_jsonl(paths["queries"])
    docs = load_jsonl(paths["docs"])
    qrels = load_jsonl(paths["qrels"])

    required_query_keys = {"query_id", "query_text"}
    required_doc_keys = {"doc_id", "text"}
    required_qrel_keys = {"query_id", "doc_id", "relevance"}

    if queries and not required_query_keys.issubset(queries[0].keys()):
        raise ValueError(f"Ungültiges Query-Schema in {name}: {queries[0].keys()}")
    if docs and not required_doc_keys.issubset(docs[0].keys()):
        raise ValueError(f"Ungültiges Doc-Schema in {name}: {docs[0].keys()}")
    if qrels and not required_qrel_keys.issubset(qrels[0].keys()):
        raise ValueError(f"Ungültiges Qrels-Schema in {name}: {qrels[0].keys()}")

    split_values = sorted(
        {str(x.get("split", "unknown")) for x in (queries + docs + qrels)}
    )

    return DatasetBundle(
        name=name,
        split_values=split_values,
        queries=queries,
        docs=docs,
        qrels=qrels,
    )


In [4]:
bundle = load_dataset(DATASET_NAME, DATASET_PATHS)

docs_by_id = {str(d["doc_id"]): d for d in bundle.docs}
doc_ids = list(docs_by_id.keys())
doc_texts = [format_passage(compose_doc_text(docs_by_id[doc_id])) for doc_id in doc_ids]
query_ids = [str(q["query_id"]) for q in bundle.queries]
query_texts = [format_query(str(q["query_text"])) for q in bundle.queries]

model = SentenceTransformer(MODEL_NAME)

doc_embeddings = model.encode(
    doc_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    convert_to_tensor=True,
)
query_embeddings = model.encode(
    query_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    convert_to_tensor=True,
)

similarity_matrix = query_embeddings @ doc_embeddings.T
max_k = max(K_VALUES)
top_scores, top_indices = similarity_matrix.topk(k=max_k, dim=1)

qrels_by_query: dict[str, set[str]] = {}
for row in bundle.qrels:
    qid = str(row["query_id"])
    did = str(row["doc_id"])
    rel = int(row["relevance"])
    if rel >= RELEVANCE_THRESHOLD and did in docs_by_id:
        qrels_by_query.setdefault(qid, set()).add(did)

query_rows: list[dict[str, Any]] = []
queries_without_qrels = 0
queries_without_relevant_docs = 0

for row_idx, query_id in enumerate(query_ids):
    query_text = str(bundle.queries[row_idx]["query_text"])
    gold_docs = qrels_by_query.get(query_id, set())
    has_qrels = any(str(r["query_id"]) == query_id for r in bundle.qrels)
    if not has_qrels:
        queries_without_qrels += 1
    if not gold_docs:
        queries_without_relevant_docs += 1

    ranked_doc_ids = [doc_ids[int(idx)] for idx in top_indices[row_idx].tolist()]
    ranked_scores = [float(score) for score in top_scores[row_idx].tolist()]

    first_rel_rank = None
    for rank_idx, doc_id in enumerate(ranked_doc_ids, start=1):
        if doc_id in gold_docs:
            first_rel_rank = rank_idx
            break

    query_rows.append(
        {
            "dataset": bundle.name,
            "query_id": query_id,
            "query_text": query_text,
            "gold_doc_ids": sorted(gold_docs),
            "top_doc_ids": ranked_doc_ids,
            "top_doc_scores": ranked_scores,
            "first_rel_rank": first_rel_rank,
        }
    )

print(f"queries={len(bundle.queries)} docs={len(bundle.docs)} qrels={len(bundle.qrels)}")


Batches: 100%|██████████| 69/69 [00:10<00:00,  6.49it/s]


queries=2204 docs=474 qrels=2204


In [5]:
metric_rows: list[dict[str, Any]] = []
for k in K_VALUES:
    recall_sum = 0.0
    success_count = 0
    mrr_sum = 0.0
    precision_sum = 0.0

    for row in query_rows:
        gold_set = set(row["gold_doc_ids"])
        retrieved = row["top_doc_ids"][:k]
        hit_count = sum(doc_id in gold_set for doc_id in retrieved)

        if gold_set:
            recall_sum += hit_count / len(gold_set)
        success_count += int(hit_count > 0)
        precision_sum += hit_count / float(k)

        rel_rank = row["first_rel_rank"]
        if rel_rank is not None and rel_rank <= k:
            mrr_sum += 1.0 / float(rel_rank)

    denom = float(len(query_rows)) if query_rows else 1.0
    metric_rows.append(
        {
            "dataset": bundle.name,
            "model": MODEL_NAME,
            "k": k,
            "recall_at_k": recall_sum / denom,
            "success_at_k": success_count / denom,
            "mrr_at_k": mrr_sum / denom,
            "precision_at_k": precision_sum / denom,
            "num_queries": len(query_rows),
        }
    )

overview_rows = [
    {
        "dataset": bundle.name,
        "split": ",".join(bundle.split_values),
        "queries": len(bundle.queries),
        "docs": len(bundle.docs),
        "qrels": len(bundle.qrels),
        "queries_without_qrels": queries_without_qrels,
        "queries_without_relevant_docs": queries_without_relevant_docs,
    }
]

sample_rows: list[dict[str, Any]] = []
for row in query_rows[:SAMPLE_QUERY_COUNT]:
    sample_rows.append(
        {
            "dataset": row["dataset"],
            "query_id": row["query_id"],
            "query_text": row["query_text"],
            "gold_doc_ids": " | ".join(row["gold_doc_ids"]),
            "top_doc_ids": " | ".join(row["top_doc_ids"]),
            "top_doc_scores": " | ".join([f"{s:.4f}" for s in row["top_doc_scores"]]),
        }
    )

overview_df = pd.DataFrame(overview_rows)
metrics_df = pd.DataFrame(metric_rows).sort_values(["dataset", "k"]).reset_index(drop=True)
samples_df = pd.DataFrame(sample_rows)

payload = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": {
        "retriever": "dense-embedding",
        "model_name": MODEL_NAME,
        "python_version": PYTHON_VERSION,
        "package": "sentence-transformers",
        "package_version": SENTENCE_TRANSFORMERS_VERSION,
        "dataset_name": DATASET_NAME,
        "split": "normalized:local",
        "corpus": "normalized docs.normalized.jsonl",
        "document_representation": "text only",
        "query_prefix": "query: ",
        "document_prefix": "passage: ",
        "chunking": "disabled; qrels are document-based",
        "similarity_metric": "cosine similarity via normalized embedding dot product",
        "batch_size": BATCH_SIZE,
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "k_values": K_VALUES,
        "relevance_threshold": RELEVANCE_THRESHOLD,
        "metrics": ["Recall@k", "Success@k", "MRR@k", "Precision@k"],
    },
    "overview": overview_rows,
    "metrics": metric_rows,
    "sample_rankings": sample_rows,
}

json_path = DENSE_OUTPUT_DIR / "dense_e5_results.json"
csv_path = DENSE_OUTPUT_DIR / "dense_e5_results.csv"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=True, indent=2)

metrics_df.to_csv(csv_path, index=False, encoding="utf-8")

print(f"JSON geschrieben: {json_path}")
print(f"CSV geschrieben: {csv_path}")
display(overview_df)
display(metrics_df)
display(samples_df)


JSON geschrieben: <PROJEKT>/code/output/dense_e5/dense_e5_results.json
CSV geschrieben: <PROJEKT>/code/output/dense_e5/dense_e5_results.csv


,dataset,split,queries,docs,qrels,queries_without_qrels,queries_without_relevant_docs
0,germanquad,local,2204,474,2204,0,0


,dataset,model,k,recall_at_k,success_at_k,mrr_at_k,precision_at_k,num_queries
0,germanquad,intfloat/multilingual-e5-large,1,0.898367,0.898367,0.898367,0.898367,2204
1,germanquad,intfloat/multilingual-e5-large,5,0.979129,0.979129,0.932214,0.195826,2204
2,germanquad,intfloat/multilingual-e5-large,10,0.986842,0.986842,0.933254,0.098684,2204


,dataset,query_id,query_text,gold_doc_ids,top_doc_ids,top_doc_scores
0,germanquad,q40369,Was kann den Verschleiß des seillosen Aufzuges...,c40369,c40369 | c36812 | c38251 | c41152 | c37727 | c...,0.8335 | 0.7969 | 0.7893 | 0.7860 | 0.7808 | 0...
1,germanquad,q40370,In welcher deutschen Stadt wird der seillose A...,c40369,c40369 | c38251 | c41474 | c36812 | c36602 | c...,0.8722 | 0.7824 | 0.7792 | 0.7771 | 0.7695 | 0...
2,germanquad,q40366,Wo wurde ein seilloser Aufzug entwickelt?,c40369,c40369 | c37731 | c38251 | c36419 | c41474 | c...,0.8613 | 0.7806 | 0.7730 | 0.7674 | 0.7671 | 0...
3,germanquad,q40367,Wie funktioniert ein seilloser Aufzug?,c40369,c40369 | c38251 | c37727 | c41391 | c36812 | c...,0.8644 | 0.8164 | 0.8042 | 0.7849 | 0.7816 | 0...
4,germanquad,q40368,Wann muss man die Zieletage in seillosen Aufzü...,c40369,c40369 | c37727 | c38601 | c38251 | c41322 | c...,0.8096 | 0.7937 | 0.7835 | 0.7833 | 0.7769 | 0...


## Interpretation des ersten Laufs

Dieses Notebook beantwortet noch nicht die komplette Modellvergleichsfrage. Es prüft zuerst, ob ein deutsch-taugliches Dense-Modell mit denselben Eingaben, denselben qrels und denselben Cutoffs wie die BM25-Baseline reproduzierbar ausgewertet werden kann.

Der nächste Schritt nach einem erfolgreichen Lauf ist deshalb nicht sofort eine breite Ergebnisinterpretation, sondern der kontrollierte Austausch des Embedding-Modells unter unveränderter Evaluationslogik.
